# Teams & Ownership

Resolve git contributors to teams, compute code ownership, identify bus factor risks.

In [ ]:
from __future__ import annotations
import warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore", category=FutureWarning)

plt.rcParams.update({"figure.figsize": (12, 6), "figure.dpi": 100, "axes.titlesize": 14})
sns.set_theme(style="whitegrid", palette="muted")

DATA_DIR = Path("../../data/processed")
PROJECT_ROOT = Path("../..")
INTERMEDIATE_DIR = Path("../../data/intermediate")

from buildanalysis.loading import BuildDataset
from buildanalysis.git import compute_file_to_target_map, compute_file_churn, compute_ownership_concentration
from buildanalysis.teams import (
    TeamConfig, resolve_git_contributors, compute_target_ownership,
    compute_file_ownership, compute_team_coupling,
)

ds = BuildDataset(DATA_DIR, intermediate_dir=INTERMEDIATE_DIR, validate=False)

## 1. Team Configuration

In [ ]:
teams_yaml = PROJECT_ROOT / "teams.yaml"
team_config = None

if teams_yaml.exists():
    try:
        team_config = TeamConfig.from_yaml(teams_yaml)
        print(f"Team configuration loaded successfully.")
        print(f"Total teams: {len(team_config.teams)}")
        print(f"Total unaffiliated members: {len(team_config.unaffiliated)}")
        print(f"Total email mappings: {len(team_config.email_to_team)}")

        for team_name, members in team_config.teams.items():
            print(f"\n{team_name}: {len(members)} member(s)")
            for member in members:
                print(f"  - {member.name}: {len(member.emails)} email(s)")
    except Exception as e:
        print(f"Error loading team config: {e}")
        print("Proceeding without team configuration.")
else:
    print(f"teams.yaml not found at {teams_yaml}")
    print("Proceeding without team configuration.")

## 2. Git Contributor Resolution

In [ ]:
if not ds.has_file("git_commit_log"):
    print("git_commit_log.parquet not available. Skipping contributor resolution.")
else:
    git_log = ds.git_commit_log

    print(f"\nGIT CONTRIBUTOR ANALYSIS")
    print(f"=" * 70)
    print(f"Total commits: {len(git_log):,}")
    print(f"Unique contributors: {git_log['contributor'].nunique()}")

    # Resolve contributors to canonical names and teams via teams.yaml
    if team_config is not None:
        resolved_log = resolve_git_contributors(git_log, team_config)
        resolved_count = resolved_log["resolved"].sum()
        total_rows = len(resolved_log)
        print(f"\nContributor resolution via teams.yaml:")
        print(f"  Resolved: {resolved_count:,} / {total_rows:,} ({100*resolved_count/total_rows:.1f}%)")
        print(f"  Unresolved: {total_rows - resolved_count:,}")

        # Show team distribution
        team_commits = resolved_log.groupby("team")["commit_hash"].nunique().sort_values(ascending=False)
        print(f"\nCommits per team:")
        for team, count in team_commits.items():
            print(f"  {team:30s} {count:6,}")
    else:
        resolved_log = git_log.copy()
        print("\nNo team config — skipping contributor resolution.")

    # Contributor concentration (HHI)
    contrib_summary = git_log["contributor"].value_counts().reset_index()
    contrib_summary.columns = ["Contributor", "Commit Count"]
    total_commits = contrib_summary["Commit Count"].sum()
    market_shares = contrib_summary["Commit Count"] / total_commits
    hhi = (market_shares ** 2).sum()

    print(f"\nContributor concentration (Herfindahl-Hirschman Index): {hhi:.4f}")
    if hhi > 0.25:
        print("  -> High concentration (top contributors dominate)")
    elif hhi > 0.10:
        print("  -> Moderate concentration")
    else:
        print("  -> Low concentration (distributed contribution)")

    print(f"\nTop 10 contributors by commit count:")
    contrib_summary["Commit %"] = (100 * contrib_summary["Commit Count"] / total_commits).round(1)
    print(contrib_summary.head(10).to_string(index=False))

## 3. Target Ownership

In [ ]:
if not ds.has_file("git_commit_log") or not ds.has_file("file_metrics"):
    print("Required data not available. Skipping target ownership analysis.")
else:
    git_log = ds.git_commit_log
    file_metrics = ds.file_metrics

    # Build file-to-target mapping using library function
    file_to_target = compute_file_to_target_map(file_metrics)

    print(f"\nTARGET OWNERSHIP ANALYSIS")
    print(f"=" * 70)

    # Compute target ownership using library function (richer output: HHI, cross-team fraction, etc.)
    if team_config is not None:
        ownership_df = compute_target_ownership(git_log, file_to_target, team_config)
        print(f"\nOwnership computed for {len(ownership_df)} targets")
        print(f"Columns: {list(ownership_df.columns)}")

        print(f"\nOwnership HHI distribution:")
        print(f"  Mean: {ownership_df['ownership_hhi'].mean():.4f}")
        print(f"  Median: {ownership_df['ownership_hhi'].median():.4f}")
        print(f"  Min: {ownership_df['ownership_hhi'].min():.4f}")
        print(f"  Max: {ownership_df['ownership_hhi'].max():.4f}")

        # Unresolved fraction analysis
        if "unresolved_fraction" in ownership_df.columns:
            high_unresolved = ownership_df[ownership_df["unresolved_fraction"] > 0.5]
            print(f"\nTargets with >50% unresolved contributors: {len(high_unresolved)}")

        # Cross-team fraction analysis
        if "cross_team_fraction" in ownership_df.columns:
            cross_team = ownership_df[ownership_df["cross_team_fraction"] > 0.3]
            print(f"Targets with >30% cross-team contributions: {len(cross_team)}")
    else:
        # Fallback: use contributor_target_commits if no team config
        if ds.has_file("contributor_target_commits"):
            ctc = ds.contributor_target_commits
            target_owner_summary = []
            for target in ctc["cmake_target"].unique():
                target_data = ctc[ctc["cmake_target"] == target]
                total_commits = target_data["commit_count"].sum()
                market_shares = target_data["commit_count"] / total_commits
                hhi = (market_shares ** 2).sum()
                sorted_commits = target_data.sort_values("commit_count", ascending=False)
                cumsum = sorted_commits["commit_count"].cumsum()
                bus_factor = (cumsum <= 0.5 * total_commits).sum() + 1
                target_owner_summary.append({
                    "cmake_target": target, "contributor_count": len(target_data),
                    "total_commits": total_commits, "ownership_hhi": hhi, "bus_factor": bus_factor,
                })
            ownership_df = pd.DataFrame(target_owner_summary)
            print(f"\nOwnership computed for {len(ownership_df)} targets (no team config)")
        else:
            ownership_df = pd.DataFrame()
            print("No contributor data available.")

    # Ownership concentration via Gini coefficient (complementary to HHI)
    gini_df = compute_ownership_concentration(git_log, file_to_target)
    if len(gini_df) > 0:
        print(f"\nOwnership Concentration (Gini coefficient):")
        print(f"  Mean Gini: {gini_df['gini'].mean():.4f}")
        print(f"  Median Gini: {gini_df['gini'].median():.4f}")
        print(f"  Targets with Gini > 0.8 (highly concentrated): {(gini_df['gini'] > 0.8).sum()}")

    # Per-file churn analysis
    file_churn = compute_file_churn(git_log)
    print(f"\nFile Churn Analysis:")
    print(f"  Total files tracked: {len(file_churn)}")
    print(f"  Mean commits per file: {file_churn['n_commits'].mean():.1f}")
    print(f"  Top 10 hotspot files:")
    for _, row in file_churn.head(10).iterrows():
        print(f"    {row['source_file']:60s} {row['n_commits']:4d} commits, {row['n_authors']:2d} authors")

## 4. Team Build Time Breakdown

In [ ]:
if team_config is None or not ds.has_file("git_commit_log") or not ds.has_file("target_metrics"):
    print("Team configuration or required data not available. Skipping team build time breakdown.")
else:
    tm = ds.target_metrics

    print(f"\nTEAM BUILD TIME BREAKDOWN")
    print(f"=" * 70)

    # Use ownership_df from previous cell (compute_target_ownership output)
    if "owning_team" in ownership_df.columns:
        team_build = ownership_df[["cmake_target", "owning_team"]].merge(
            tm[["cmake_target", "total_build_time_ms"]], on="cmake_target", how="left"
        )
        team_build["total_build_time_ms"] = team_build["total_build_time_ms"].fillna(0)

        team_summary = team_build.groupby("owning_team")["total_build_time_ms"].sum().reset_index()
        team_summary.columns = ["Team", "Build Time (ms)"]
        team_summary["Build Time (sec)"] = (team_summary["Build Time (ms)"] / 1000).round(2)
        team_summary = team_summary.sort_values("Build Time (ms)", ascending=False)

        print("\nBuild time contribution by owning team:")
        print(team_summary[["Team", "Build Time (sec)"]].to_string(index=False))

        if len(team_summary) > 0:
            fig, ax = plt.subplots(figsize=(10, 5))
            ax.barh(team_summary["Team"], team_summary["Build Time (sec)"])
            ax.set_xlabel("Build Time (seconds)")
            ax.set_title("Build Time Contribution by Team")
            plt.tight_layout()
            plt.show()
    else:
        print("Ownership data does not contain owning_team column. Skipping.")

    # Per-file ownership drill-down
    if "file_to_target" in dir():
        file_ownership = compute_file_ownership(git_log, team_config, file_to_target=file_to_target)
        if len(file_ownership) > 0:
            print(f"\nPer-File Ownership (top 15 cross-team files):")
            if "cross_team_fraction" in file_ownership.columns:
                cross_team_files = file_ownership.nlargest(15, "cross_team_fraction")
                display_cols = [c for c in ["source_file", "owning_team", "cross_team_fraction", "contributor_count"]
                               if c in cross_team_files.columns]
                print(cross_team_files[display_cols].to_string(index=False))

## 5. Team Coupling

In [ ]:
if team_config is None or not ds.has_file("edge_list") or "ownership_df" not in dir() or len(ownership_df) == 0:
    print("Team configuration or required data not available. Skipping team coupling analysis.")
else:
    edge_list = ds.edge_list

    print(f"\nTEAM COUPLING ANALYSIS")
    print(f"=" * 70)

    # Use compute_team_coupling from teams.py (richer output: edge_count, public_edge_count, target_pairs)
    coupling_df = compute_team_coupling(edge_list, ownership_df)

    if len(coupling_df) > 0:
        print(f"\nInter-team dependencies (top 20):")
        top_couplings = coupling_df.sort_values("edge_count", ascending=False).head(20)
        for _, row in top_couplings.iterrows():
            public_note = f" ({row['public_edge_count']} public)" if "public_edge_count" in row.index else ""
            print(f"  {row['source_team']:30} -> {row['dest_team']:30} : {row['edge_count']:3} edges{public_note}")

        # Build coupling matrix for heatmap
        teams_list = sorted(set(coupling_df["source_team"]) | set(coupling_df["dest_team"]))
        coupling_matrix = pd.DataFrame(0, index=teams_list, columns=teams_list)
        for _, row in coupling_df.iterrows():
            if row["source_team"] in coupling_matrix.index and row["dest_team"] in coupling_matrix.columns:
                coupling_matrix.loc[row["source_team"], row["dest_team"]] += row["edge_count"]

        print(f"\nCoupling matrix (team x team dependency count):")
        print(coupling_matrix)

        if len(teams_list) > 1 and len(teams_list) <= 10:
            fig, ax = plt.subplots(figsize=(10, 8))
            sns.heatmap(coupling_matrix, annot=True, fmt="d", cmap="YlOrRd", ax=ax)
            ax.set_title("Team Dependency Coupling Matrix")
            plt.tight_layout()
            plt.show()
    else:
        print("No inter-team dependencies found.")

## 5b. Contributor Clustering (when unresolved fraction is high)

When many contributors are not mapped to teams, discover organic team structure from commit patterns.

In [ ]:
UNRESOLVED_THRESHOLD = 0.20  # trigger contributor clustering when >20% unresolved

run_clustering = False
if "ownership_df" in dir() and "unresolved_fraction" in ownership_df.columns:
    avg_unresolved = ownership_df["unresolved_fraction"].mean()
    if avg_unresolved > UNRESOLVED_THRESHOLD:
        run_clustering = True
        print(f"Average unresolved fraction: {avg_unresolved:.1%} (> {UNRESOLVED_THRESHOLD:.0%} threshold)")
        print("Running contributor clustering to discover organic team structure...")
elif team_config is None:
    run_clustering = True
    print("No team config available — running contributor clustering as primary team discovery.")

if run_clustering and ds.has_file("contributor_target_commits"):
    from buildanalysis.contributors import (
        build_contributor_target_matrix, cluster_contributors_hierarchical,
        compute_ownership, compute_bus_factor,
    )

    ctc = ds.contributor_target_commits

    # Build contributor-target matrix
    matrix = build_contributor_target_matrix(ctc, min_contributor_commits=10, min_target_commits=5)
    print(f"\nContributor-target matrix: {matrix.shape[0]} contributors x {matrix.shape[1]} targets")

    if matrix.shape[0] >= 3:
        # Cluster contributors hierarchically
        cluster_result = cluster_contributors_hierarchical(matrix, cut_levels=[3, 5, 8])
        groups_df = cluster_result["assignments"]

        # Use the cut level that produces the most balanced groups
        group_col = groups_df.columns[-1]  # last column = finest cut
        n_groups = groups_df[group_col].nunique()
        print(f"Discovered {n_groups} contributor groups (hierarchical clustering)")

        group_sizes = groups_df[group_col].value_counts()
        print(f"\nGroup sizes:")
        for gid, size in group_sizes.items():
            members = groups_df[groups_df[group_col] == gid]["contributor"].tolist()
            print(f"  Group {gid}: {size} contributors — {', '.join(members[:5])}")

        # Compute ownership scores per group
        ownership_scores = compute_ownership(ctc, groups_df)
        bus_factor = compute_bus_factor(ctc, groups_df)

        if len(bus_factor) > 0:
            low_bus = bus_factor[bus_factor.get("active_contributors", bus_factor.columns[-1]) <= 1] if "active_contributors" in bus_factor.columns else pd.DataFrame()
            if len(low_bus) > 0:
                print(f"\nBus factor warning: {len(low_bus)} targets with single active contributor group")

        # Save for downstream use
        ds.save_intermediate("contributor_groups", groups_df)
        print(f"\nSaved contributor_groups.parquet ({len(groups_df)} rows)")
    else:
        print("Too few contributors for clustering (need at least 3).")
elif not run_clustering:
    print(f"Unresolved fraction below threshold — skipping contributor clustering.")
else:
    print("contributor_target_commits not available — skipping contributor clustering.")

## 6. Save Intermediates

In [ ]:
# Save target_ownership intermediate (consumed by NB08 for recommendations)
if "ownership_df" in dir() and len(ownership_df) > 0:
    path = ds.save_intermediate("target_ownership", ownership_df)
    print(f"Saved target_ownership to {path} ({len(ownership_df)} rows, columns: {list(ownership_df.columns)})")

# Save Gini-based concentration metrics
if "gini_df" in dir() and len(gini_df) > 0:
    path = ds.save_intermediate("ownership_concentration", gini_df)
    print(f"Saved ownership_concentration to {path} ({len(gini_df)} rows)")

# Save file churn
if "file_churn" in dir() and len(file_churn) > 0:
    path = ds.save_intermediate("file_churn", file_churn)
    print(f"Saved file_churn to {path} ({len(file_churn)} rows)")

print("\nNotebook completed successfully.")